# V2 — Half-Spread Settlement Control (measure-only re-aggregation; no new scoring)

**Designation (fixed, verbatim):** half-spread settlement control (measure-only re-aggregation; no new scoring)

**Research question.** Does the calm-bar (low-activity-tercile) concentration of the same-row
macro-F1 edge sit where Roll (1984) bid-ask bounce is mechanically capable of producing it
(half-spread comparable to or above the frozen +/-3.0 bps no-trade band), or does it persist
where bounce is mechanically too small (half-spread well below the band)?

**What this notebook does NOT do:** no model refit, no re-scoring, no reselection, no operating
point, no clean-test claim, no tradeability claim. It re-slices the FROZEN per-row prediction
dumps by a raw-bar-derived conditioning variable. Whatever the outcome, only the INTERPRETATION
of the activity-tercile map changes (limitation resolved or confirmed) — never the headline claims.


## Pre-Registration Summary (frozen)

Full document: `docs/protocols/v2_halfspread_control_preregistration_20260701.md` — committed
BEFORE any dump contact. Frozen choices: Roll (1984) half-spread `sqrt(-autocov_lag1)` of
within-day 5-minute log close returns, trailing 21 observed trading days ending at d-1 (day d
excluded), pinned covariance denominator n-1, minimum 400 pooled pairs; non-negative
autocovariance is its own named cell (`roll_undefined_nonneg_autocov`), never imputed. Partition
of eligible rows by `half-spread / (3.0 bps)`: `<=0.5`, `(0.5,1]`, `(1,2]`, `>2` plus the two
named undefined cells; coarse anchors `<=1` / `>1`. Discriminating table = same-row model vs
stratified-dummy macro-F1 delta per cell inside the LOW activity tercile; two frozen seeds
(101, 202), per-seed rows + seed-mean; per-seed per-(ticker, trading_day) block-bootstrap
intervals (1000 draws, seed 12345) as DESCRIPTIVE context only. Predeclared outcomes:
consistent_with_bounce_domination / not_explainable_by_bounce_alone / inconclusive — evaluated
mechanically by `microstructure.verdict_from_readout` (validation domain = the registered
verdict; guarded domain = non-independent replication context, NEVER pooled).


## Expected Artifacts (per domain run)

```text
/content/lst_models_results/v2_halfspread_control/<run_id>/
├── run_manifest.json                      # domain, contact flags, config hash, dump hashes, verdict
├── artifact_inventory.csv
├── halfspread_day_spread.csv              # per (ticker, trading_day) proxy + cells (auditable)
├── halfspread_partition_readout.csv       # R1: all eligible rows, per cell x seed + seed-mean
├── halfspread_low_tercile_readout.csv     # R2: THE discriminating table (low activity tercile)
├── halfspread_occupancy.csv               # R3: cell x tercile row/day counts
├── halfspread_autocov_by_tercile.csv      # R4: lag-1 autocov/autocorr per tercile (H2 signature)
├── halfspread_cs_robustness_readout.csv   # R6: Corwin-Schultz robustness (no verdict power)
└── halfspread_verdict.json                # mechanical predeclared verdict + fired conditions
```

Durable copy: `My Drive/lst_models/results/v2_halfspread_control/<run_id>/` (+ `drive_backup_manifest.json`).
Two separate runs (validation first, then optionally guarded) — separate run folders, never pooled.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import copy
import importlib
import json
import subprocess
import sys
import zipfile

RUN_PROJECT_BOOTSTRAP = True
PROJECT_BOOTSTRAP_MODE = "github_commit"  # github_commit | manual_upload | already_present

PROJECT_REPO_URL = "https://github.com/zkc768/lstm-zhang.git"
PROJECT_REPO_COMMIT = "<fill_with_full_bundle_commit>"  # must contain this notebook + config + prereg + src + tests
PROJECT_ROOT = Path("/content/lst_models")

# Heavy switches: OFF by default. Run the VALIDATION domain first (primary).
RUN_RAW_DATA_SYNC = True
RUN_UPSTREAM_DRIVE_SYNC = True
RUN_HALFSPREAD_VALIDATION = False
RUN_HALFSPREAD_GUARDED = False
RUN_DRIVE_BACKUP = True
RUN_ARTIFACT_DISPLAY = False

STAGE_NAME = "v2_halfspread_control"
ROUTE = "lst_models"
SCOPE = "halfspread_control_measure_only"
DESIGNATION = "half-spread settlement control (measure-only re-aggregation; no new scoring)"
FROZEN_SEEDS = [101, 202]
BAND_BPS = 3.0

STAGE00_RUN_ID = "20260610_051705_347450"
STAGE01_RUN_ID = "20260610_075002"
STAGE03_RUN_ID = "20260610_133305_716174"
V2_1_RUN_ID = "20260618_063559_889276"

# Frozen guarded dump integrity (from the frozen addenda manifests
# artifacts/05_guarded_activity_tercile/guarded_activity_tercile_manifest.json).
EXPECTED_GUARDED_DUMP_SHA256 = {
    "v2_1_predictions.csv": "6481f7958834b2c58cf3217ccbf7274ed25ec70cc176fd1b3955888317910300",
    "v2_1_baseline_predictions.csv": "cd6925e1dfb0d5305b212586dfe84c2dde7d7cba244a036b3b61df7cac46fcdb",
}

STAGE00_OUTPUT_DIR = Path("/content/lst_models_results/00_data_split_label_freeze") / STAGE00_RUN_ID
STAGE01_OUTPUT_DIR = Path("/content/lst_models_results/01_feature_window_search") / STAGE01_RUN_ID
STAGE03_OUTPUT_DIR = Path("/content/lst_models_results/03_frozen_validation_readout") / STAGE03_RUN_ID
V2_1_OUTPUT_DIR = Path("/content/lst_models_results/v2_1_guarded_walkforward_readout") / V2_1_RUN_ID
OUTPUT_DIR = Path("/content/lst_models_results/v2_halfspread_control")
RAW_DATA_DIR = Path("/content/lst_models_raw_stock_data")

STAGE00_DRIVE_PATH_PARTS = ["lst_models", "results", "00_data_split_label_freeze", STAGE00_RUN_ID]
STAGE01_DRIVE_PATH_PARTS = ["lst_models", "results", "01_feature_window_search", STAGE01_RUN_ID]
STAGE03_DRIVE_PATH_PARTS = ["lst_models", "results", "03_frozen_validation_readout", STAGE03_RUN_ID]
V2_1_DRIVE_PATH_PARTS = ["lst_models", "results", "v2_1_guarded_walkforward_readout", V2_1_RUN_ID]
HALFSPREAD_DRIVE_RESULT_PATH_PARTS = ["lst_models", "results", "v2_halfspread_control"]

def quote_drive_query_value(value):
    return str(value).replace("\\", "\\\\").replace("'", "\\'")

def run_cmd(args, cwd=None):
    print("+", " ".join(str(a) for a in args))
    subprocess.run(args, cwd=cwd, check=True)

def looks_like_project_root(path):
    return (
        (path / "configs" / "stages" / "v2_halfspread_control.yaml").exists()
        and (path / "docs" / "protocols" / "v2_halfspread_control_preregistration_20260701.md").exists()
        and (path / "src" / "lst_models" / "stages" / "halfspread_control.py").exists()
        and (path / "src" / "lst_models" / "microstructure.py").exists()
    )

def safe_extract_project_zip(zip_path):
    destination = Path("/content").resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            member_path = Path(member.filename)
            if member_path.is_absolute() or ".." in member_path.parts:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
            target = (destination / member_path).resolve()
            if target != destination and destination not in target.parents:
                raise ValueError(f"Unsafe path in uploaded zip: {member.filename}")
        archive.extractall(destination)
    for candidate in [Path("/content/lst_models"), Path("/content") / zip_path.stem]:
        if looks_like_project_root(candidate):
            return candidate
    raise FileNotFoundError("No lst_models project root found after zip extraction.")

def upload_and_extract_project_zip():
    try:
        from google.colab import files
    except ImportError as exc:
        raise RuntimeError("manual_upload mode only works inside Colab.") from exc
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.endswith(".zip")]
    if not zip_names:
        raise ValueError("Upload a .zip file containing the lst_models project folder.")
    return safe_extract_project_zip(Path("/content") / zip_names[0])

if RUN_PROJECT_BOOTSTRAP:
    if PROJECT_BOOTSTRAP_MODE == "github_commit":
        if "<" in PROJECT_REPO_COMMIT:
            raise ValueError("fill PROJECT_REPO_COMMIT with the half-spread full-bundle commit before bootstrapping")
        if (PROJECT_ROOT / ".git").exists():
            run_cmd(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        else:
            run_cmd(["git", "clone", PROJECT_REPO_URL, str(PROJECT_ROOT)])
            run_cmd(["git", "checkout", PROJECT_REPO_COMMIT], cwd=PROJECT_ROOT)
        actual_commit = subprocess.check_output(
            ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
        ).strip()
        assert actual_commit == PROJECT_REPO_COMMIT, (actual_commit, PROJECT_REPO_COMMIT)
    elif PROJECT_BOOTSTRAP_MODE == "manual_upload":
        PROJECT_ROOT = upload_and_extract_project_zip()
    elif PROJECT_BOOTSTRAP_MODE == "already_present":
        pass
    else:
        raise ValueError("PROJECT_BOOTSTRAP_MODE must be github_commit, manual_upload, or already_present")

STAGE_CONFIG_PATH = PROJECT_ROOT / "configs" / "stages" / "v2_halfspread_control.yaml"
PREREGISTRATION_PATH = PROJECT_ROOT / "docs" / "protocols" / "v2_halfspread_control_preregistration_20260701.md"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks" / "v2_halfspread_control_colab.ipynb"
STAGE_ENTRYPOINT_PATH = PROJECT_ROOT / "src" / "lst_models" / "stages" / "halfspread_control.py"
DOMAIN_MODULE_PATH = PROJECT_ROOT / "src" / "lst_models" / "microstructure.py"
RAW_DATA_MANIFEST_PATH = PROJECT_ROOT / "configs" / "lst_models_data.yaml"
REQUIRED_PROJECT_FILES = [
    STAGE_CONFIG_PATH, PREREGISTRATION_PATH, NOTEBOOK_PATH,
    STAGE_ENTRYPOINT_PATH, DOMAIN_MODULE_PATH, RAW_DATA_MANIFEST_PATH,
]
missing_project_files = [path for path in REQUIRED_PROJECT_FILES if not path.exists()]
if missing_project_files:
    missing_text = "\n".join(f"- {path}" for path in missing_project_files)
    raise FileNotFoundError("half-spread control bootstrap target is missing required files:\n" + missing_text)

SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

def clear_project_import_cache():
    cached = [name for name in sys.modules if name == "lst_models" or name.startswith("lst_models.")]
    for name in cached:
        del sys.modules[name]
    importlib.invalidate_caches()

clear_project_import_cache()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROJECT_COMMIT:", PROJECT_REPO_COMMIT)
print("STAGE_NAME:", STAGE_NAME)
print("DESIGNATION:", DESIGNATION)
print("RUN_HALFSPREAD_VALIDATION:", RUN_HALFSPREAD_VALIDATION)
print("RUN_HALFSPREAD_GUARDED:", RUN_HALFSPREAD_GUARDED)


## Config Load, Runtime Injection, And Contract Check

The repo config is UN-ARMED (`run_domain: null`). This cell loads it, injects the Colab runtime
paths, and asserts the frozen pre-registration constants. The per-domain configs are armed only
inside the run cells below.


In [ ]:
try:
    import yaml
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("PyYAML is required to read the half-spread config.") from exc

def require_path(path):
    if not path.exists():
        raise FileNotFoundError(f"missing required file: {path}")

require_path(STAGE_CONFIG_PATH)
require_path(PREREGISTRATION_PATH)
require_path(RAW_DATA_MANIFEST_PATH)

with STAGE_CONFIG_PATH.open("r", encoding="utf-8") as handle:
    halfspread_config = yaml.safe_load(handle)

assert halfspread_config["stage_name"] == STAGE_NAME
assert halfspread_config["route"] == ROUTE
assert halfspread_config["scope"] == SCOPE
assert halfspread_config["run_domain"] is None, "repo config must stay un-armed (run_domain null)"
assert int(halfspread_config["new_scoring_events"]) == 0
assert int(halfspread_config["official_validation_scoring_events"]) == 0
assert halfspread_config["no_final_model_selected"] is True
assert halfspread_config["v2_frozen_selection_unchanged"] is True

estimator = halfspread_config["estimator"]
assert estimator["primary"] == "roll_1984_halfspread"
assert int(estimator["roll_window_days"]) == 21
assert int(estimator["roll_min_pairs"]) == 400
assert float(estimator["band_bps"]) == BAND_BPS
assert [float(edge) for edge in estimator["ratio_edges"]] == [0.5, 1.0, 2.0]

readout = halfspread_config["readout"]
assert readout["seeds"] == FROZEN_SEEDS
assert readout["primary_model_row"] == "tcn_frozen_primary"
assert readout["candidate_id"] == "price_volume_time_w20"
assert int(readout["verdict_min_rows"]) == 5000
assert int(readout["bootstrap"]["iterations"]) == 1000
assert int(readout["bootstrap"]["seed"]) == 12345

label_policy = halfspread_config["label_policy"]
assert label_policy["label_config_id"] == "h09_bps3p0"
assert float(label_policy["no_trade_band_bps"]) == BAND_BPS
assert int(label_policy["horizon_k"]) == 9

domains = halfspread_config["domains"]
val = domains["validation"]
assert val["holdout_test_contact"] is False
assert val["stage00_run_id"] == STAGE00_RUN_ID
assert val["stage01_run_id"] == STAGE01_RUN_ID
assert val["stage03_run_id"] == STAGE03_RUN_ID
assert int(val["expected_dump_rows"]) == 302128
gua = domains["guarded"]
assert gua["holdout_test_contact"] is True
assert gua["holdout_contact_tier"] == "guarded_historically_contacted"
assert gua["clean_test_claim"] is False
assert gua["v2_1_run_id"] == V2_1_RUN_ID

# runtime path injection (contract: computed paths go INTO the config before run_stage)
val["stage00_runtime_run_dir"] = str(STAGE00_OUTPUT_DIR)
val["stage01_runtime_run_dir"] = str(STAGE01_OUTPUT_DIR)
val["stage03_runtime_run_dir"] = str(STAGE03_OUTPUT_DIR)
gua["v2_1_runtime_run_dir"] = str(V2_1_OUTPUT_DIR)
halfspread_config["inputs"]["raw_data_dir"] = str(RAW_DATA_DIR)
halfspread_config["inputs"]["raw_data_manifest"] = str(RAW_DATA_MANIFEST_PATH)
halfspread_config["inputs"]["notebook_path"] = str(NOTEBOOK_PATH)
halfspread_config["outputs"]["output_dir"] = str(OUTPUT_DIR)

forbidden_wording = list(halfspread_config["forbidden"]["wording"])
assert forbidden_wording, "forbidden wording list must be non-empty"
assert "clean test" in forbidden_wording

expected_output_names = {
    "manifest": "run_manifest.json",
    "artifact_inventory": "artifact_inventory.csv",
    "day_spread": "halfspread_day_spread.csv",
    "partition_readout": "halfspread_partition_readout.csv",
    "low_tercile_readout": "halfspread_low_tercile_readout.csv",
    "occupancy": "halfspread_occupancy.csv",
    "autocov_by_tercile": "halfspread_autocov_by_tercile.csv",
    "cs_robustness_readout": "halfspread_cs_robustness_readout.csv",
    "verdict": "halfspread_verdict.json",
}
for output_key, output_name in expected_output_names.items():
    assert halfspread_config["outputs"][output_key] == output_name

print(json.dumps({
    "stage_name": halfspread_config["stage_name"],
    "scope": halfspread_config["scope"],
    "run_domain": halfspread_config["run_domain"],
    "estimator": estimator["primary"],
    "roll_window_days": estimator["roll_window_days"],
    "band_bps": estimator["band_bps"],
    "seeds": readout["seeds"],
}, indent=2))


## Upstream Inputs: Frozen Run Folders And Raw Data (Drive file IDs; no folder scanning)

Validation domain needs Stage 00 / 01 / 03 artifacts (including the 302,128-row
`03_validation_predictions.csv`) plus the five raw `.txt` files. Guarded domain needs the V2.1
run folder (`v2_1_predictions.csv` ~569 MB + `v2_1_baseline_predictions.csv` ~118 MB — allow
several minutes) plus the raw files. Guarded dump sha256 is verified against the frozen addenda
manifests before any measurement.


In [ ]:
from lst_models.artifacts import require_artifacts
from lst_models.config import hash_file

def get_drive_service():
    try:
        from google.colab import auth
        from googleapiclient.discovery import build
    except ImportError as exc:
        raise RuntimeError("Drive access only works inside Colab with Google API dependencies.") from exc
    auth.authenticate_user()
    return build("drive", "v3")

def find_unique_drive_child(service, parent_id, name, mime_type=None):
    escaped_name = quote_drive_query_value(name)
    query_parts = [f"name = '{escaped_name}'", f"'{parent_id}' in parents", "trashed = false"]
    if mime_type:
        query_parts.append(f"mimeType = '{mime_type}'")
    response = service.files().list(q=" and ".join(query_parts), spaces="drive",
                                    fields="files(id, name, mimeType, size)", pageSize=10).execute()
    matches = response.get("files", [])
    if len(matches) != 1:
        raise FileNotFoundError(f"expected exactly one Drive item named {name!r} under parent {parent_id}; found {len(matches)}")
    return matches[0]

def resolve_drive_folder(service, path_parts):
    folder_id = "root"
    folder_mime = "application/vnd.google-apps.folder"
    for folder_name in path_parts:
        folder = find_unique_drive_child(service, folder_id, folder_name, folder_mime)
        folder_id = folder["id"]
    return folder_id

def download_drive_file(service, file_id, output_path):
    from googleapiclient.http import MediaIoBaseDownload
    output_path.parent.mkdir(parents=True, exist_ok=True)
    request = service.files().get_media(fileId=file_id)
    with output_path.open("wb") as handle:
        downloader = MediaIoBaseDownload(handle, request)
        done = False
        while not done:
            _, done = downloader.next_chunk()

def ensure_drive_folder(service, parent_id, name):
    folder_mime = "application/vnd.google-apps.folder"
    escaped_name = quote_drive_query_value(name)
    query = f"name = '{escaped_name}' and '{parent_id}' in parents and trashed = false and mimeType = '{folder_mime}'"
    matches = service.files().list(q=query, fields="files(id, name)", pageSize=10).execute().get("files", [])
    if len(matches) == 1:
        return matches[0]["id"]
    if len(matches) > 1:
        raise RuntimeError(f"Duplicate Drive folders named {name!r} under parent {parent_id}")
    created = service.files().create(body={"name": name, "mimeType": folder_mime, "parents": [parent_id]},
                                     fields="id, name").execute()
    return created["id"]

def ensure_drive_path(service, path_parts):
    folder_id = "root"
    for part in path_parts:
        folder_id = ensure_drive_folder(service, folder_id, part)
    return folder_id

def find_drive_children(service, parent_id, name):
    escaped_name = quote_drive_query_value(name)
    query = f"name = '{escaped_name}' and '{parent_id}' in parents and trashed = false"
    return service.files().list(q=query, fields="files(id, name, mimeType, size)", pageSize=10).execute().get("files", [])

def upload_or_update_drive_file(service, parent_id, local_path, display_name=None):
    from googleapiclient.http import MediaFileUpload
    name = display_name or local_path.name
    matches = find_drive_children(service, parent_id, name)
    media = MediaFileUpload(str(local_path), resumable=True)
    if len(matches) == 0:
        uploaded = service.files().create(body={"name": name, "parents": [parent_id]},
                                          media_body=media, fields="id, name, size").execute()
        action = "uploaded"
    elif len(matches) == 1:
        uploaded = service.files().update(fileId=matches[0]["id"], media_body=media,
                                          fields="id, name, size").execute()
        action = "updated"
    else:
        raise RuntimeError(f"Duplicate Drive files named {name!r} under parent {parent_id}")
    print(f"{action}: {name}")
    return dict(uploaded)

def sync_stage_artifacts_from_drive(service, output_dir, path_parts, required_names, label):
    run_folder_id = resolve_drive_folder(service, path_parts)
    for artifact_name in required_names:
        output_path = Path(output_dir) / artifact_name
        if output_path.exists():
            continue
        file_meta = find_unique_drive_child(service, run_folder_id, artifact_name)
        download_drive_file(service, file_meta["id"], output_path)
        print(f"downloaded {label}: {artifact_name}")

def sync_raw_data_from_drive(service):
    with RAW_DATA_MANIFEST_PATH.open("r", encoding="utf-8") as handle:
        raw_manifest = yaml.safe_load(handle)
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
    for ticker in raw_manifest["tickers"]:
        spec = raw_manifest["raw_source"]["files"][ticker]
        output_path = RAW_DATA_DIR / spec["name"]
        if output_path.exists():
            continue
        download_drive_file(service, spec["file_id"], output_path)
        print(f"downloaded raw data: {output_path.name}")

service = None
if RUN_HALFSPREAD_VALIDATION or RUN_HALFSPREAD_GUARDED:
    service = get_drive_service()
    if RUN_RAW_DATA_SYNC:
        sync_raw_data_from_drive(service)

if RUN_HALFSPREAD_VALIDATION:
    if RUN_UPSTREAM_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, STAGE00_OUTPUT_DIR, STAGE00_DRIVE_PATH_PARTS,
                                        val["required_stage00_artifacts"], "stage00")
        sync_stage_artifacts_from_drive(service, STAGE01_OUTPUT_DIR, STAGE01_DRIVE_PATH_PARTS,
                                        val["required_stage01_artifacts"], "stage01")
        sync_stage_artifacts_from_drive(service, STAGE03_OUTPUT_DIR, STAGE03_DRIVE_PATH_PARTS,
                                        val["required_stage03_artifacts"], "stage03")
    require_artifacts(STAGE00_OUTPUT_DIR, val["required_stage00_artifacts"])
    require_artifacts(STAGE01_OUTPUT_DIR, val["required_stage01_artifacts"])
    require_artifacts(STAGE03_OUTPUT_DIR, val["required_stage03_artifacts"])
    print("validation-domain upstream inputs verified.")

if RUN_HALFSPREAD_GUARDED:
    if RUN_UPSTREAM_DRIVE_SYNC:
        sync_stage_artifacts_from_drive(service, V2_1_OUTPUT_DIR, V2_1_DRIVE_PATH_PARTS,
                                        gua["required_v2_1_artifacts"], "v2_1")
    require_artifacts(V2_1_OUTPUT_DIR, gua["required_v2_1_artifacts"])
    for dump_name, expected_sha in EXPECTED_GUARDED_DUMP_SHA256.items():
        observed_sha = hash_file(V2_1_OUTPUT_DIR / dump_name)
        if observed_sha != expected_sha:
            raise ValueError(
                f"guarded dump integrity failure for {dump_name}: expected {expected_sha}, got {observed_sha}"
            )
        print(f"sha256 verified: {dump_name}")
    print("guarded-domain upstream inputs verified.")

if not (RUN_HALFSPREAD_VALIDATION or RUN_HALFSPREAD_GUARDED):
    print("both domain switches are False; no inputs synced.")


## Run 1 — VALIDATION Domain (primary; `holdout_test_contact=false`)

Reads the frozen Stage 03 dump, rebuilds the windowed train labels through the frozen mechanism
chain (raw bars -> features -> windows, count-gated), replays the stratified dummy under the
dual 1e-9 equality gates (reconstruction-or-nothing), computes the Roll proxy on train+validation
bars only, and writes the partition readouts + the registered verdict. Expect roughly Stage-04-like
runtime (tens of minutes: feature rebuild + 1000-draw block bootstrap per cell x seed).


In [ ]:
result_validation = None
if RUN_HALFSPREAD_VALIDATION:
    from lst_models.stages.halfspread_control import run_stage

    config_validation = copy.deepcopy(halfspread_config)
    config_validation["run_domain"] = "validation"
    result_validation = run_stage(config_validation)

    validation_run_id = result_validation.run_dir.name
    print("VALIDATION RUN_ID:", validation_run_id)
    print("RUN_DIR:", result_validation.run_dir)
    verdict_record = json.loads(result_validation.verdict_record.read_text(encoding="utf-8"))
    print("verdict:", verdict_record["verdict"])
    print("role:", verdict_record["role"])
    for f in sorted(result_validation.run_dir.iterdir()):
        print(f"  {f.name} ({f.stat().st_size:,} bytes)")
else:
    print("RUN_HALFSPREAD_VALIDATION=False; validation domain not executed.")


## Durable Drive Result Save — Validation Run


In [ ]:
from lst_models.stages.halfspread_control import REQUIRED_HALFSPREAD_ARTIFACTS

def backup_halfspread_run(result, expected_domain):
    run_dir = result.run_dir
    run_id = run_dir.name
    missing = [name for name in REQUIRED_HALFSPREAD_ARTIFACTS if not (run_dir / name).exists()]
    if missing:
        raise FileNotFoundError(f"missing required artifacts before Drive backup in {run_dir}: {missing}")
    manifest_data = json.loads(result.run_manifest.read_text(encoding="utf-8"))
    if manifest_data.get("run_domain") != expected_domain:
        raise ValueError(f"backup refused: manifest run_domain != {expected_domain}")
    if int(manifest_data.get("new_scoring_events", -1)) != 0:
        raise ValueError("backup refused: manifest must record new_scoring_events=0")
    if manifest_data.get("no_final_model_selected") is not True:
        raise ValueError("backup refused: manifest must record no_final_model_selected=true")
    if expected_domain == "validation" and manifest_data.get("holdout_test_contact") is not False:
        raise ValueError("backup refused: validation manifest must record holdout_test_contact=false")
    if expected_domain == "guarded" and manifest_data.get("holdout_contact_tier") != "guarded_historically_contacted":
        raise ValueError("backup refused: guarded manifest must record the guarded contact tier")

    drive_path_parts = HALFSPREAD_DRIVE_RESULT_PATH_PARTS + [run_id]
    drive_folder_id = ensure_drive_path(service, drive_path_parts)
    backup_manifest_path = run_dir / "drive_backup_manifest.json"
    local_files = sorted(p for p in run_dir.rglob("*") if p.is_file() and p.name != backup_manifest_path.name)
    uploads = []
    for path in local_files:
        uploaded = upload_or_update_drive_file(service, drive_folder_id, path)
        uploaded["bytes"] = path.stat().st_size
        uploads.append(uploaded)
    backup_manifest = {
        "stage_name": STAGE_NAME,
        "run_id": run_id,
        "stage_run_id": run_id,
        "run_domain": expected_domain,
        "verdict": manifest_data.get("verdict"),
        "local_output_dir": str(run_dir),
        "drive_path": "My Drive/" + "/".join(drive_path_parts),
        "drive_path_parts": drive_path_parts,
        "drive_folder_id": drive_folder_id,
        "uploaded_file_names": [u["name"] for u in uploads] + [backup_manifest_path.name],
        "uploaded_files": uploads + [{"name": backup_manifest_path.name, "bytes": None, "self_reference": True}],
        "sync_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "new_scoring_events": 0,
        "no_final_model_selected": True,
    }
    backup_manifest_path.write_text(json.dumps(backup_manifest, indent=2), encoding="utf-8")
    upload_or_update_drive_file(service, drive_folder_id, backup_manifest_path)
    print("stage_run_id:", run_id)
    print("drive_path:", backup_manifest["drive_path"])
    print("drive_folder_id:", drive_folder_id)
    return backup_manifest

if RUN_DRIVE_BACKUP and RUN_HALFSPREAD_VALIDATION:
    if result_validation is None:
        raise RuntimeError("RUN_DRIVE_BACKUP=True requires the validation run to have executed.")
    backup_halfspread_run(result_validation, "validation")
else:
    print("validation backup skipped (switches off or run not executed).")


## Run 2 — GUARDED Domain (secondary; non-independent; NEVER pooled with validation)

Same measure-only re-aggregation on the frozen V2.1 dumps. Inherits the V2.1 designation
"guarded, historically-contacted walk-forward readout" (`holdout_contact_tier=
guarded_historically_contacted`, `clean_test_claim=false`). Its verdict field is replication
CONTEXT only — the registered verdict comes from the validation run.


In [ ]:
result_guarded = None
if RUN_HALFSPREAD_GUARDED:
    from lst_models.stages.halfspread_control import run_stage as run_stage_guarded

    config_guarded = copy.deepcopy(halfspread_config)
    config_guarded["run_domain"] = "guarded"
    result_guarded = run_stage_guarded(config_guarded)

    guarded_run_id = result_guarded.run_dir.name
    print("GUARDED RUN_ID:", guarded_run_id)
    print("RUN_DIR:", result_guarded.run_dir)
    verdict_record = json.loads(result_guarded.verdict_record.read_text(encoding="utf-8"))
    print("verdict (replication context, non-independent):", verdict_record["verdict"])
    for f in sorted(result_guarded.run_dir.iterdir()):
        print(f"  {f.name} ({f.stat().st_size:,} bytes)")
else:
    print("RUN_HALFSPREAD_GUARDED=False; guarded domain not executed.")


## Durable Drive Result Save — Guarded Run


In [ ]:
if RUN_DRIVE_BACKUP and RUN_HALFSPREAD_GUARDED:
    if result_guarded is None:
        raise RuntimeError("RUN_DRIVE_BACKUP=True requires the guarded run to have executed.")
    backup_halfspread_run(result_guarded, "guarded")
else:
    print("guarded backup skipped (switches off or run not executed).")


## Readout Display (compact; per domain, never pooled)


In [ ]:
if RUN_ARTIFACT_DISPLAY:
    import pandas as pd

    for label, result in (("validation", result_validation), ("guarded", result_guarded)):
        if result is None:
            print(f"{label}: not executed")
            continue
        print("=" * 70)
        print(f"{label} run: {result.run_dir.name}  (domains reported separately, never pooled)")
        low = pd.read_csv(result.run_dir / "halfspread_low_tercile_readout.csv")
        display(low[low["seed"] == "seed_mean"][
            ["cell", "n_rows", "macro_f1", "dummy_macro_f1", "delta_vs_dummy", "thin_cell"]
        ])
        occupancy = pd.read_csv(result.run_dir / "halfspread_occupancy.csv")
        display(occupancy.pivot_table(index="cell", columns="activity_tercile",
                                      values="n_rows", aggfunc="sum", fill_value=0))
        verdict = json.loads((result.run_dir / "halfspread_verdict.json").read_text(encoding="utf-8"))
        print("verdict:", verdict["verdict"], "| role:", verdict["role"])
else:
    print("RUN_ARTIFACT_DISPLAY=False; display skipped.")


## Honest Interpretation And Limitations

- This is a **half-spread settlement control (measure-only re-aggregation; no new scoring)**.
  It selects no model, marks no operating point, and makes no clean-test, significance, or
  tradeability statement. Block-bootstrap intervals are descriptive context only.
- The verdict in `halfspread_verdict.json` is the MECHANICAL application of the pre-registered
  rules (pre-registration section 8). The registered verdict is the VALIDATION run's; the guarded
  run is non-independent replication context. The two are never pooled.
- Outcome wording is frozen: "consistent with bounce-domination" confirms and sharpens the
  existing calm-bar limitation; "not explainable by bounce alone" resolves it toward genuine
  conditional structure; "inconclusive" keeps the ledger's open-item wording plus occupancy facts.
  In every case, only the interpretation of the activity-tercile map changes — headline same-row
  dummy deltas, the guarded designation, and the no-model-selected stance are untouched.
- The Roll proxy is an effective-spread PROXY from 5-minute closes (no quotes, no trade signs);
  "activity" remains the eligible-row-count proxy, not volume or liquidity.
- After the runs, the user (not this notebook) may append the ledger row / budget-ledger contact
  entries per the templates in the pre-registration section 3 — canonical ledger edits are
  out of scope for this notebook.
